### Data Cleaning 

This notebook performs structured data cleaning on extracted tennis datasets.
1. checking duplicates 
2. null values 
3. cleaning identifiers
4. renaming columns
5. optimizing data types
6. saving the cleaned data for further use.

In [1]:
# Import required libraries for data cleaning and transformation
import requests
import json
import pandas as pd
from dotenv import load_dotenv
import os

In [ ]:
# Setting the base path where raw extracted files are stored
data_path = '../data/'

In [ ]:
# Load raw extracted datasets generated during API extraction phase

category_df = pd.read_csv(os.path.join(data_path, 'raw_file/category.csv'))
competition_df = pd.read_csv(os.path.join(data_path, 'raw_file/competition.csv'))
complex_df = pd.read_csv(os.path.join(data_path, 'raw_file/complex.csv'))
venue_df = pd.read_csv(os.path.join(data_path, 'raw_file/venues.csv'))
ranking_df = pd.read_csv(os.path.join(data_path, 'raw_file/Competitor_Rankings.csv'))
competitor_df = pd.read_csv(os.path.join(data_path, 'raw_file/competitor.csv'))

#### 1. Handle null values
#Checking for missing values in each dataset


In [4]:
category_df.isna().sum()

id      0
name    0
dtype: int64

No Null Values in category table.

In [9]:
competition_df.isna().sum()
competition_df['gender'] = competition_df['gender'].fillna('unknown') # Filling missing gender values with 'unknown'
competition_df['parent_id'] = competition_df['parent_id'].fillna('ROOT') # Filling missing parent_id with 'ROOT' (top-level competitions)

In [10]:
complex_df.isna().sum()

complex_id    0
name          0
dtype: int64

In [11]:
venue_df.isna().sum()

id              0
name            0
complex_id      0
city_name       0
country_name    0
country_code    0
timezone        0
dtype: int64

In [12]:
ranking_df.isna().sum()

rank                   0
movement               0
points                 0
competitions_played    0
competitor_id          0
gender                 0
year                   0
week                   0
tour                   0
dtype: int64

 No Null Values in complex,venue and ranking table

In [13]:
competitor_df.isna().sum()

id               0
name             0
country          0
country_code    57
abbreviation     0
dtype: int64

In [14]:
competitor_df[competitor_df['country_code'].isnull()].head(5) # Checking rows where country_code is missing

,id,name,country,country_code,abbreviation
10,sr:competitor:163504,"Medvedev, Daniil",Neutral,NaN,MED
15,sr:competitor:90080,"Khachanov, Karen",Neutral,NaN,KHA
17,sr:competitor:106755,"Rublev, Andrey",Neutral,NaN,RUB
188,sr:competitor:124930,"Safiullin, Roman",Neutral,NaN,SAF
225,sr:competitor:686709,"Simakin, Ilia",Neutral,NaN,SIM


####  2. Check for duplicates
#Removing duplicate rows from each dataset to ensure data consistency

In [15]:
#category_df
category_df.drop_duplicates(inplace = True)
category_df.shape

(18, 2)

In [16]:
#competition_df 
competition_df.drop_duplicates(inplace = True)
competition_df.shape

(6477, 6)

In [17]:
#complex_df
complex_df.drop_duplicates(inplace = True)
complex_df.shape

(760, 2)

In [18]:
#venue_df
venue_df.drop_duplicates(inplace = True)

In [19]:
#ranking_df
ranking_df.drop_duplicates(inplace=True)

In [20]:
#competitor_df
competitor_df.drop_duplicates(inplace=True)

### 3. Clean columns
#Removing unnecessary prefixes from ID columns for standardization

In [21]:
#category_df
category_df['id'] = category_df['id'].str[12:]

In [22]:

# competition_df
competition_df['id'] = competition_df['id'].str[15:]
competition_df['parent_id'] = competition_df['parent_id'].apply(lambda x: x[15:] if x != 'ROOT' else 'ROOT')
competition_df['category_id'] = competition_df['category_id'].str[12:]

In [23]:
#complex_df
complex_df['complex_id'] = complex_df['complex_id'].str[11:]
complex_df

,complex_id,name
0,705,Nacional
1,1078,Estadio la Cartuja
2,1495,Sibur Arena
3,2375,Complexo de Tenis do Jamor
4,4032,Shree Shiv Chhatrapati Sports Complex
...,...,...
755,86800,Lsu Tennis Complex
756,86838,2 Rue du Bonheur
757,86946,Lake Las Vegas Sports Club
758,87022,Fujairah Tennis & Country Club


In [24]:
#venue_df
venue_df['id'] = venue_df['id'].str[9:]
venue_df['complex_id'] = venue_df['complex_id'].str[11:]
venue_df

,id,name,complex_id,city_name,country_name,country_code,timezone
0,70045,Cancha Central,705,Santiago,CHILE,CHL,America/Santiago
1,74858,Court One,1078,Seville,SPAIN,ESP,Europe/Madrid
2,74856,Centre Court,1078,Seville,SPAIN,ESP,Europe/Madrid
3,62153,TC Dynamo,1495,Saint Petersburg,RUSSIAN FEDERATION,RUS,Europe/Moscow
4,1500,CENTER COURT,1495,Saint Petersburg,RUSSIAN FEDERATION,RUS,Europe/Moscow
...,...,...,...,...,...,...,...
3889,87046,Centre Court,87022,Fujairah,UNITED ARAB EMIRATES,ARE,Asia/Dubai
3890,87050,Court 2,87022,Fujairah,UNITED ARAB EMIRATES,ARE,Asia/Dubai
3891,87086,Tennis Center Murska Sobota - Court 3,87066,Murska Sobota,SLOVENIA,SVN,Europe/Ljubljana
3892,87084,Tennis Center Murska Sobota - Court 2,87066,Murska Sobota,SLOVENIA,SVN,Europe/Ljubljana


In [25]:
#ranking_df
ranking_df['id'] = range(1, len(ranking_df) + 1)
ranking_df['competitor_id'] = ranking_df['competitor_id'].str[14:] 
ranking_df

,rank,movement,points,competitions_played,competitor_id,gender,year,week,tour,id
0,1,0,13550,18,407573,men,2026,9,ATP,1
1,2,0,10400,19,225050,men,2026,9,ATP,2
2,3,0,5280,18,14882,men,2026,9,ATP,3
3,4,0,4555,22,57163,men,2026,9,ATP,4
4,5,0,4405,22,359602,men,2026,9,ATP,5
...,...,...,...,...,...,...,...,...,...,...
995,496,-5,111,16,214264,women,2026,9,WTA,996
996,497,-5,111,21,677415,women,2026,9,WTA,997
997,498,2,110,26,315623,women,2026,9,WTA,998
998,499,-5,110,9,819750,women,2026,9,WTA,999


In [26]:
#competitor_df
competitor_df['id'] = competitor_df['id'].str[14:]
competitor_df['country_code'] = competitor_df['country_code'].fillna('unkown')# Filling missing country_code with 'unknown'

#### 5. Assign correct datatype

In [27]:
venue_df.info() # Checking current data types

<class 'pandas.DataFrame'>
RangeIndex: 3894 entries, 0 to 3893
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            3894 non-null   str  
 1   name          3894 non-null   str  
 2   complex_id    3894 non-null   str  
 3   city_name     3894 non-null   str  
 4   country_name  3894 non-null   str  
 5   country_code  3894 non-null   str  
 6   timezone      3894 non-null   str  
dtypes: str(7)
memory usage: 213.1 KB


In [28]:
# Optimizing competition dataset
competition_dtypes = {'type': 'category', 'gender': 'category'}
competition_df = competition_df.astype(competition_dtypes)
competition_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6477 entries, 0 to 6476
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   id           6477 non-null   str     
 1   name         6477 non-null   str     
 2   parent_id    6477 non-null   str     
 3   type         6477 non-null   category
 4   gender       6477 non-null   category
 5   category_id  6477 non-null   str     
dtypes: category(2), str(4)
memory usage: 215.4 KB


In [29]:
# Optimizing venue dataset
venue_dtypes = {'city_name': 'category', 'country_name': 'category', 'country_code': 'category', 'timezone': 'category'}
venue_df = venue_df.astype(venue_dtypes)
venue_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3894 entries, 0 to 3893
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   id            3894 non-null   str     
 1   name          3894 non-null   str     
 2   complex_id    3894 non-null   str     
 3   city_name     3894 non-null   category
 4   country_name  3894 non-null   category
 5   country_code  3894 non-null   category
 6   timezone      3894 non-null   category
dtypes: category(4), str(3)
memory usage: 115.7 KB


In [30]:
# Optimizing ranking dataset
ranking_dtype = {'gender': 'category', 'tour' : 'category'}
ranking_df = ranking_df.astype(ranking_dtype)
ranking_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   rank                 1000 non-null   int64   
 1   movement             1000 non-null   int64   
 2   points               1000 non-null   int64   
 3   competitions_played  1000 non-null   int64   
 4   competitor_id        1000 non-null   str     
 5   gender               1000 non-null   category
 6   year                 1000 non-null   int64   
 7   week                 1000 non-null   int64   
 8   tour                 1000 non-null   category
 9   id                   1000 non-null   int64   
dtypes: category(2), int64(7), str(1)
memory usage: 64.7 KB


In [31]:
# Optimizing competitor dataset
competitor_dtype = {'country': 'category', 'country_code' : 'category', 'abbreviation': 'category'}
competitor_df = competitor_df.astype(competitor_dtype)
competitor_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   id            1000 non-null   str     
 1   name          1000 non-null   str     
 2   country       1000 non-null   category
 3   country_code  1000 non-null   category
 4   abbreviation  1000 non-null   category
dtypes: category(3), str(2)
memory usage: 25.9 KB


In [32]:
# Creating clean_data folder if it does not exist
os.makedirs(os.path.join(data_path, 'clean_data'), exist_ok=True)

#### 6. Save into CSV

In [33]:
# Saving cleaned CSV files for further analysis or database storage

competition_df.to_csv(os.path.join(data_path, 'clean_data/competition.csv'), index = False)
category_df.to_csv(os.path.join(data_path, 'clean_data/category.csv'), index = False)
venue_df.to_csv(os.path.join(data_path, 'clean_data/venue.csv'), index = False)
complex_df.to_csv(os.path.join(data_path, 'clean_data/complex.csv'), index = False)
ranking_df.to_csv(os.path.join(data_path, 'clean_data/ranking.csv'), index = False)
competitor_df.to_csv(os.path.join(data_path, 'clean_data/competitor.csv'), index = False)